# Inspect generated tomograms and masks

Compare raw and noisy tomograms with instance labels for any run produced
by `run.py` or `simulation_demo.ipynb`. This notebook reads results without
modifying masks. It can optionally export a figure and selected TIFF slices.

Start with `uv run --extra notebooks jupyter lab`, then set `RUN_DIR` below.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src").is_dir()
)
os.chdir(PROJECT_ROOT)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pptools.io import ZarrReader


## Choose a run and slice axis

In [ ]:
RUN_DIR = Path("data/GOH_6_250/outputs/example/sim_res_0")
AXIS = "z"
SLICE_INDEX = None  # None selects the middle slice.
EXPORT = False
EXPORT_DIR = RUN_DIR / "previews"


In [ ]:
volumes = {
    name: ZarrReader(str(RUN_DIR / f"{name}.zarr")).get_dask_array()
    for name in ("tomo", "tomo_noisy", "mask")
}
assert len({volume.shape for volume in volumes.values()}) == 1
for name, volume in volumes.items():
    print(f"{name}: shape={volume.shape}, dtype={volume.dtype}")

axis = {"z": 0, "y": 1, "x": 2}[AXIS]
size = volumes["tomo"].shape[axis]
index = size // 2 if SLICE_INDEX is None else SLICE_INDEX
assert 0 <= index < size, f"Choose a slice between 0 and {size - 1}."
selection = [slice(None)] * 3
selection[axis] = index
slices = {name: volume[tuple(selection)].compute() for name, volume in volumes.items()}


## Compare aligned slices

Instance colors repeat every 20 IDs; they show particle separation rather
than a quantitative label scale. Zero is displayed as background. Change
`AXIS` and `SLICE_INDEX` above to inspect other locations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, name in zip(axes[:2], ("tomo", "tomo_noisy")):
    ax.imshow(slices[name], cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
mask_slice = slices["mask"]
axes[2].imshow(np.ma.masked_where(mask_slice == 0, mask_slice % 20), cmap="tab20", vmin=0, vmax=19, interpolation="nearest")
axes[2].set_title("Instance mask (colors repeat)")
for ax in axes:
    ax.axis("off")
fig.suptitle(f"{RUN_DIR.name}: {AXIS}={index}")
fig.tight_layout()
plt.show()
print(f"Nonzero particle IDs in this slice: {len(np.unique(mask_slice[mask_slice > 0]))}")


## Optional export

Set `EXPORT = True` to save the comparison PNG and one TIFF slice per volume.
TIFF output preserves each slice's dtype and original mask IDs. This exports
2D slices, not the full 3D stores; repeated exports replace matching filenames.

In [ ]:
if EXPORT:
    import tifffile

    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    suffix = f"{AXIS}_{index:04d}"
    fig.savefig(EXPORT_DIR / f"comparison_{suffix}.png", dpi=150, bbox_inches="tight")
    for name, image in slices.items():
        tifffile.imwrite(EXPORT_DIR / f"{name}_{suffix}.tiff", image)
    print(f"Exported previews to {EXPORT_DIR.resolve()}")
